# Query Foundry IQ con il Python SDK

Interroga una knowledge base esistente con `KnowledgeBaseRetrievalClient` di Azure AI Search e visualizza contenuti, riferimenti e attività. Il notebook usa per default l'ambiente azd selezionato; può anche leggere gli output ARM dei notebook di deploy.

**Prerequisiti:** Python 3.10+, Azure CLI autenticata, permesso **Search Index Data Reader** e un ambiente già configurato. Installa le [dipendenze](../requirements.txt) e riavvia il kernel se hai aggiornato pacchetti già importati. Il notebook usa `azure-search-documents==12.2.0b1` e l'API `2026-11-01-preview`.

**Effetti della query:** la KB può chiamare modelli e fonti esterne, con relativi costi. La fonte MCP può inizializzare o ricaricare il catalogo SQL se `SQL_AUTO_SEED=true`; controlla la [configurazione MCP](../../mcp-ricambi/README.md) prima di eseguire la query.

Per i dettagli dell'SDK vedi la [guida Microsoft alla retrieve](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python).

In [1]:
CONFIG_SOURCE = "azd"             # "azd", "env" (notebooks/demo/.env) oppure "arm"
ENV_FILE = ""                      # solo con "env"; vuoto = notebooks/demo/.env
AZD_ENVIRONMENT_NAME = ""          # vuoto = ambiente azd attualmente selezionato
SUBSCRIPTION_ID = ""               # opzionale; con azd deve coincidere con l'ambiente
ENVIRONMENT_NAME = "wpc-demo"      # usato solo per il deployment ARM
DEPLOYMENT_NAME = f"wpc-foundryiq-{ENVIRONMENT_NAME}"

KB_PROFILE = "multi-fonte"         # "multi-fonte" oppure "acl"
KB_OVERRIDE = ""                   # opzionale: nome di una KB diversa
QUERY_MODE = "auto"                # auto, messages o intents
FORWARD_USER_TOKEN = True           # ACL attive; con multi-fonte il token va comunque a SharePoint remoto
QUESTION = "Ogni quanto si sostituiscono le membrane della Graphene Vapor Reclaim Tank e quante GRT-002 sono disponibili?"

## 1 · Configurazione e identità

`azd` fornisce endpoint e nomi senza stampare o conservare altre impostazioni. Con `CONFIG_SOURCE = "arm"` vengono letti gli output del deployment indicato. Subscription e tenant sono verificati senza cambiare la selezione globale di Azure CLI.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: serve infra/main.bicep.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env, load_env_file
if str(ROOT / "webapp-chat") not in sys.path:
    sys.path.insert(0, str(ROOT / "webapp-chat"))
from mcp_references import notebook_reference_rows, summarize_references
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata: installala ed esegui 'az login'.")


def az(*args, output=True, check=True):
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_azd_values(environment_name):
    return load_azd_env(environment_name, root=ROOT, override=True)


def load_env_values(env_file):
    return load_env_file(env_file or ROOT / "notebooks" / "demo" / ".env", override=True)


def load_outputs(deployment_name, subscription_id):
    deployment = az("deployment", "sub", "show", "--name", deployment_name, "--subscription", subscription_id)
    if not deployment:
        raise RuntimeError(f"Deployment ARM '{deployment_name}' non trovato nella subscription selezionata.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


def load_configuration(required):
    if CONFIG_SOURCE in ("azd", "env"):
        if CONFIG_SOURCE == "azd":
            out = load_azd_values(AZD_ENVIRONMENT_NAME)
            where = "dell'ambiente azd"
            source = f"ambiente azd {out.get('AZURE_ENV_NAME') or AZD_ENVIRONMENT_NAME or '(corrente)'}"
        else:
            out = load_env_values(ENV_FILE)
            where = "del file .env"
            source = f"file {ENV_FILE or 'notebooks/demo/.env'}"
        selected = out.get("AZURE_SUBSCRIPTION_ID")
        if not isinstance(selected, str) or not selected.strip():
            raise RuntimeError(f"AZURE_SUBSCRIPTION_ID mancante nella configurazione {where}.")
        if SUBSCRIPTION_ID and SUBSCRIPTION_ID.casefold() != selected.casefold():
            raise RuntimeError(f"SUBSCRIPTION_ID non corrisponde alla subscription {where}.")
        account = az("account", "show", "--subscription", selected)
        if account["id"].casefold() != selected.casefold():
            raise RuntimeError(f"Azure CLI ha restituito una subscription diversa da quella {where}.")
    elif CONFIG_SOURCE == "arm":
        account = az("account", "show", *(["--subscription", SUBSCRIPTION_ID] if SUBSCRIPTION_ID else []))
        out = load_outputs(DEPLOYMENT_NAME, account["id"])
        source = f"deployment ARM {DEPLOYMENT_NAME}"
    else:
        raise ValueError("CONFIG_SOURCE deve essere 'azd', 'env' oppure 'arm'.")
    missing = [key for key in ["AZURE_TENANT_ID", "AZURE_RESOURCE_GROUP", *required]
               if not isinstance(out.get(key), str) or not out[key].strip()]
    if missing:
        raise RuntimeError(f"Valori mancanti in {source}: {', '.join(missing)}.")
    if out["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
        raise RuntimeError("Il tenant della configurazione non corrisponde alla subscription selezionata in Azure CLI.")
    return out, account, source


out, account, source = load_configuration(["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"])
SUBSCRIPTION_ID = account["id"]
SEARCH_ENDPOINT = out["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"].rstrip("/")
if KB_PROFILE not in ("multi-fonte", "acl"):
    raise ValueError("KB_PROFILE deve essere 'multi-fonte' oppure 'acl'.")
kb_key, default_kb = (("FOUNDRYIQ_DEMO_KB_NAME", "wpc-foundryiq-kb") if KB_PROFILE == "multi-fonte"
                      else ("FOUNDRYIQ_ACL_KB_NAME", "wpc-acl-kb"))
KB_NAME = KB_OVERRIDE or out.get(kb_key) or default_kb
MODE = ("messages" if KB_PROFILE == "multi-fonte" else "intents") if QUERY_MODE == "auto" else QUERY_MODE
if MODE not in ("messages", "intents"):
    raise ValueError("QUERY_MODE deve essere 'auto', 'messages' oppure 'intents'.")
# remoteSharePoint (policies-sharepoint-ks) interroga sempre SharePoint come utente: senza token fallisce.
SEND_USER_TOKEN = FORWARD_USER_TOKEN or KB_PROFILE == "multi-fonte"
if SEND_USER_TOKEN and not FORWARD_USER_TOKEN:
    print("ACL disattivate, ma il token utente viene inoltrato perché la KB include SharePoint remoto.")
print(f"Configurazione: {source} | KB: {KB_NAME} | modalità: {MODE} | token utente: {SEND_USER_TOKEN}")

## 2 · Richiesta SDK

Il profilo multi-fonte usa `messages` con pianificazione della KB; il profilo ACL usa un intento semantico perché la KB della demo ha reasoning `minimal`. Se usi una KB diversa, scegli la modalità compatibile con la sua configurazione.

Con `FORWARD_USER_TOKEN=True`, Azure CLI deve rappresentare un utente: il suo token viene inoltrato per applicare le autorizzazioni delle fonti. Con `False` sul profilo `acl` la query avviene senza identità utente e le ACL non restituiscono documenti. Il profilo multi-fonte inoltra comunque il token: la knowledge source `remoteSharePoint` lo richiede sempre e senza header restituisce `x-ms-query-source-authorization is invalid, null or empty`.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

try:
    sdk_version = version("azure-search-documents")
except PackageNotFoundError as exc:
    raise RuntimeError("Installa le dipendenze del notebook, incluso azure-search-documents==12.2.0b1.") from exc
if sdk_version != "12.2.0b1":
    raise RuntimeError(f"Azure Search SDK {sdk_version}: installa azure-search-documents==12.2.0b1 e riavvia il kernel.")

import pandas as pd
from IPython.display import display
from azure.identity import AzureCliCredential
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
)

SEARCH_SCOPE = "https://search.azure.com/.default"
SEARCH_API_VERSION = "2026-11-01-preview"


def build_request(question, mode):
    if not question or not question.strip():
        raise ValueError("Inserisci una domanda non vuota in QUESTION.")
    if mode == "messages":
        return KnowledgeBaseRetrievalRequest(
            messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
            include_activity=True, max_runtime_in_seconds=120,
        )
    if mode == "intents":
        return KnowledgeBaseRetrievalRequest(
            intents=[KnowledgeRetrievalSemanticIntent(search=question)],
            include_activity=True, max_runtime_in_seconds=120,
        )
    raise ValueError("La modalità della query deve essere 'messages' oppure 'intents'.")


def retrieve_question(question, mode, forward_user, endpoint, kb_name, subscription_id, account):
    request = build_request(question, mode)
    if forward_user and (account.get("user") or {}).get("type") != "user":
        raise RuntimeError("Per inoltrare il token sorgente esegui 'az login' con un utente, non con un service principal.")
    with AzureCliCredential(subscription=subscription_id) as credential:
        user_token = credential.get_token(SEARCH_SCOPE).token if forward_user else None
        with KnowledgeBaseRetrievalClient(endpoint=endpoint, knowledge_base_name=kb_name, credential=credential,
                                          api_version=SEARCH_API_VERSION) as client:
            return client.retrieve(retrieval_request=request, query_source_authorization=user_token)

## 3 · Esegui la query

Modifica `QUESTION` nella prima cella e riesegui questa cella e le successive. Il token è usato solo in memoria e non viene mostrato.

In [ ]:
result = retrieve_question(QUESTION, MODE, SEND_USER_TOKEN, SEARCH_ENDPOINT, KB_NAME, SUBSCRIPTION_ID, account)
print(f"Retrieve completata: {len(result.references or [])} riferimenti, {len(result.activity or [])} attività.")

## 4 · Contenuto, riferimenti e attività

La risposta rispetta l'output configurato nella KB: con `extractiveData` contiene dati recuperati, non una risposta sintetizzata. L'attività può includere errori o avvisi delle singole fonti anche quando la chiamata è riuscita. Le chiamate MCP senza riferimenti restano nella tabella delle attività; gli argomenti possono contenere dati riservati.

In [ ]:
texts = [content.text for message in result.response or [] for content in message.content or []
         if getattr(content, "text", None)]
if texts:
    print("\n\n".join(texts))
else:
    print("Nessun contenuto testuale restituito dalla knowledge base.")

In [ ]:
payload = result.as_dict()
references = notebook_reference_rows(summarize_references(payload))
activity_rows = []
for item in payload.get("activity") or []:
    mcp = item.get("mcpServerArguments") or {}
    activity_rows.append({"id": item.get("id"), "tipo": item.get("type"), "fonte": item.get("knowledgeSourceName"),
                          "tool MCP": mcp.get("toolName"), "argomenti MCP": mcp.get("toolArguments"),
                          "risultati": item.get("count"), "ms": item.get("elapsedMs"),
                          "inizio": item.get("startedAt"), "fine": item.get("completedAt"),
                          "errore": item.get("error"), "avviso": item.get("warning")})
with pd.option_context("display.max_colwidth", None, "display.max_columns", None, "display.max_rows", None):
    if references:
        display(pd.DataFrame(references))
    else:
        print("Nessun riferimento restituito.")
    if activity_rows:
        display(pd.DataFrame(activity_rows))
    else:
        print("Nessuna attività restituita.")

Le celle possono mostrare contenuti riservati accessibili all'utente. Prima di condividere o committare il notebook, rimuovi gli output (`Clear All Outputs`). Un risultato assente, da solo, non dimostra che sia stato escluso dalle ACL.